# Algoritmi Greedy: Activity Selection

Notebook pentru demonstrarea strategiei care selecteaza numarul maxim de activitati compatibile. Executati celulele in ordine.

**Ideea Greedy:** alegem activitatea compatibila care se termina cel mai devreme.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'main.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from main import Activity, select_activities, is_compatible_selection

## 1. Enuntul problemei

Fiecare activitate are un moment de inceput si unul de final. Vrem sa selectam cat mai multe activitati care nu se suprapun. Consideram intervale semi-deschise: daca o activitate se termina exact cand incepe urmatoarea, ele sunt compatibile.

## Date de intrare

Datele sunt definite direct in urmatorul cell Python, prin obiecte `Activity`. Valorile sunt hardcodate pentru ca demonstratia sa poata fi rerulata identic, fara introducere manuala de la tastatura.


In [ ]:
activities = [
    Activity('A', 1, 4),
    Activity('B', 3, 5),
    Activity('C', 0, 6),
    Activity('D', 5, 7),
    Activity('E', 8, 9),
    Activity('F', 5, 9),
]

print('Date de intrare:')
for activity in activities:
    print(f'{activity.name}: [{activity.start}, {activity.finish})')


## 2. Aplicarea strategiei Greedy

Sortam dupa timpul de finalizare, alegem prima activitate, apoi acceptam o activitate numai daca incepe dupa sau exact la finalul ultimei activitati selectate.

In [ ]:
selected = select_activities(activities)

print('Activitati selectate:')
for activity in selected:
    print(f'{activity.name}: [{activity.start}, {activity.finish})')

print(f'\nNumar activitati selectate: {len(selected)}')
print('Selectia este compatibila:', is_compatible_selection(selected))

Rezultatul asteptat pentru acest set este **A, D, E**. Algoritmul obtine 3 activitati. Sortarea are costul dominant: O(n log n); parcurgerea dupa sortare este O(n).

## 3. De ce nu alegem pur si simplu activitatea care incepe prima?

Un interval care incepe primul poate ocupa aproape tot timpul si poate bloca mai multe activitati scurte, dar compatibile intre ele.

### Exemplu separat: cand alegerea dupa timpul de inceput esueaza

Datele de intrare sunt definite intr-un cell separat. Activitatea A incepe prima, dar ocupa aproape tot intervalul disponibil.


In [ ]:
bad_for_earliest_start = [
    Activity('A', 1, 10),
    Activity('B', 2, 3),
    Activity('C', 3, 4),
    Activity('D', 4, 5),
]


In [ ]:
def select_by_earliest_start(items):
    ordered = sorted(items, key=lambda a: (a.start, a.finish, a.name))
    result = []
    last_finish = None
    for activity in ordered:
        if last_finish is None or activity.start >= last_finish:
            result.append(activity)
            last_finish = activity.finish
    return result

print('Greedy dupa start:', [a.name for a in select_by_earliest_start(bad_for_earliest_start)])
print('Greedy dupa finish:', [a.name for a in select_activities(bad_for_earliest_start)])


## 4. Contraexemplu: durata minima

Activitatea S este mai scurta, dar se suprapune cu ambele activitati A si B. Alegerea ei reduce numarul total de activitati.

### Date de intrare pentru contraexemplul „durata minima”

Activitatea S este cea mai scurta, dar se suprapune cu A si B. Datele sunt definite separat de codul care aplica cele doua criterii.


In [ ]:
bad_for_shortest = [
    Activity('A', 0, 4),
    Activity('B', 5, 9),
    Activity('S', 3, 6),
]


In [ ]:
def select_by_shortest_duration(items):
    ordered = sorted(items, key=lambda a: (a.finish - a.start, a.finish, a.name))
    result = []
    for activity in ordered:
        if is_compatible_selection(result + [activity]):
            result.append(activity)
    return result

print('Greedy dupa durata:', [a.name for a in select_by_shortest_duration(bad_for_shortest)])
print('Greedy dupa finish:', [a.name for a in select_activities(bad_for_shortest)])


## 5. Alta problema: 0/1 Knapsack

In varianta 0/1, fiecare obiect este luat integral sau deloc. Vom compara euristica raportului valoare/greutate cu rezultatul exact pentru un exemplu mic.

### Date de intrare pentru 0/1 Knapsack

Capacitatea rucsacului este 50. Obiectele sunt indivizibile: fiecare este luat integral sau deloc. Datele sunt introduse in cell-ul urmator.


In [ ]:
from main import Item

items = [
    Item('A', weight=10, value=60),
    Item('B', weight=20, value=100),
    Item('C', weight=30, value=120),
]
capacity = 50

print('Obiecte:')
for item in items:
    print(f'{item.name}: greutate={item.weight}, valoare={item.value}, raport={item.value / item.weight:.1f}')
print('Capacitate:', capacity)


In [ ]:
from main import (
    greedy_by_value_density,
    optimal_by_bruteforce,
    total_value,
    total_weight,
)

greedy_result = greedy_by_value_density(items, capacity)
exact_result = optimal_by_bruteforce(items, capacity)

print('Greedy:', [item.name for item in greedy_result],
      '| greutate =', total_weight(greedy_result),
      '| valoare =', total_value(greedy_result))
print('Optim:', [item.name for item in exact_result],
      '| greutate =', total_weight(exact_result),
      '| valoare =', total_value(exact_result))


**Concluzie:** Greedy este eficient si optim pentru problema Activity Selection neponderata, datorita proprietatii alegerii Greedy. Dar strategia trebuie justificata separat pentru fiecare problema. Contraexemplul 0/1 Knapsack arata ca raportul valoare/greutate nu garanteaza o solutie optima cand obiectele nu pot fi fractionate.